# ARC NeuroGolf static ONNX solver

Reference layout adapted from the uploaded fill/additive-marking notebook. The task-specific modelling cell uses a semantic feature-tree or a symbolic reflection builder, not raw output-template lookup.

In [1]:
!rm -rf /kaggle/working/*
%reset -f

In [2]:
COMPETITION = '/kaggle/input/competitions/neurogolf-2026'

In [3]:
import importlib.util, subprocess, sys
missing=[p for p in ['onnx','onnxruntime','onnxscript','torch','numpy'] if importlib.util.find_spec(p) is None]
if missing:
    subprocess.check_call([sys.executable,'-m','pip','install','-q',*missing])
print('dependencies ok')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 79.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 722.0/722.0 kB 34.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.8/166.8 kB 10.3 MB/s eta 0:00:00
dependencies ok


In [4]:
import json, os, time, hashlib, zipfile,  csv, base64
import glob, sys,math, random, collections,io,shutil
from pathlib import Path
import numpy as np
import torch
import onnx
import onnxruntime as ort
import torch, torch.nn as nn, torch.nn.functional as F
from collections import defaultdict
from onnx import shape_inference

In [5]:

TASK_ID='task080'; CH=10; H=30; W=30

ROOT=Path(COMPETITION)
if not (ROOT/f'{TASK_ID}.json').exists():
    ROOT=Path('/mnt/data')
TASK_PATH=ROOT/f'{TASK_ID}.json'

OUT_DIR=Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd()
ONNX_PATH=OUT_DIR/f'{TASK_ID}_static_graph.onnx'
ZIP_PATH=OUT_DIR/f'{TASK_ID}_zero_pad_static_graph_submission.zip'
GENERIC_ZIP=OUT_DIR/'submission.zip'
AUDIT_JSON=OUT_DIR/f'{TASK_ID}_zero_pad_audit.json'
AUDIT_CSV=OUT_DIR/f'{TASK_ID}_zero_pad_audit.csv'


In [6]:

def grid_to_tensor(grid):
    x=np.zeros((1,CH,H,W),dtype=np.float32)
    a=np.array(grid,dtype=np.int64)
    for r in range(min(H,a.shape[0])):
        for c in range(min(W,a.shape[1])):
            v=int(a[r,c])
            if 0<=v<CH:
                x[0,v,r,c]=1.0
    return x

def padded_target(grid):
    tgt=np.zeros((H,W),dtype=np.int64)
    a=np.array(grid,dtype=np.int64)
    tgt[:min(H,a.shape[0]),:min(W,a.shape[1])]=a[:H,:W]
    return tgt

def tensor_to_grid(y):
    y=np.asarray(y)
    if y.ndim==4:
        y=y[0]
    return y.argmax(axis=0).astype(np.int64).tolist()

def exact_eval(sess, examples):
    exact=0; used=0; skipped=0; first=None
    inp=sess.get_inputs()[0].name
    for i,ex in enumerate(examples):
        if len(ex['input'])>H or len(ex['input'][0])>W or len(ex['output'])>H or len(ex['output'][0])>W:
            skipped += 1
            continue
        y=sess.run(None,{inp:grid_to_tensor(ex['input'])})[0]
        pred=np.asarray(y)[0].argmax(axis=0).astype(np.int64)
        out=padded_target(ex['output'])
        ok=np.array_equal(pred,out)
        exact += int(ok); used += 1
        if not ok and first is None:
            first={'idx':i,'wrong_pixels':int((pred!=out).sum()),
                   'first_wrong_pixels':np.argwhere(pred!=out)[:20].tolist()}
    return {'exact':exact,'total':used,'skipped_over_30':skipped,'first_wrong':first}

def valid_30_examples(examples):
    return [ex for ex in examples
            if len(ex['input'])<=H and len(ex['input'][0])<=W and len(ex['output'])<=H and len(ex['output'][0])<=W]

def op_counts(model):
    return dict(collections.Counter(n.op_type for n in model.graph.node))

def onnx_shape(value_info):
    dims=[]
    for d in value_info.type.tensor_type.shape.dim:
        dims.append(int(d.dim_value) if d.dim_value else None)
    return dims

class Task080StaticSafe(nn.Module):
    """Task080 grid-cell pattern propagation, exported as a pure static ONNX graph.

    Real rule at cell-grid level:
    - the image is a grid of solid cells separated by full separator lines;
    - one marker colour appears both inside a local motif and as isolated marker cells;
    - copy the motif around every isolated marker of the same colour, clipping at borders;
    - keep the ONNX interface fixed at [1,10,30,30] and keep padded area zero.

    This version avoids Shape/Gather/ConstantOfShape/Expand/Range/ScatterND in addition
    to the required forbidden ops. Those shape-construction ops can be accepted by ONNX
    checker but fail or score zero in stricter Kaggle static-graph runners.
    """
    def __init__(self):
        super().__init__()
        for N in [6,8,10]:
            self.register_buffer(f'z1_row_{N}', torch.zeros(1,1,1,N))
            self.register_buffer(f'z1_col_{N}', torch.zeros(1,1,N,1))
        self.register_buffer('z10_col', torch.zeros(1,10,30,1))
        self.register_buffer('z10_row31', torch.zeros(1,10,1,31))
        self.register_buffer('z9_29_1', torch.zeros(1,9,29,1))
        self.register_buffer('z9_1_30', torch.zeros(1,9,1,30))
        self.register_buffer('w2', torch.ones(9,1,2,2))
        self.register_buffer('w3', torch.ones(9,1,3,3))
        self.register_buffer('w4', torch.ones(9,1,4,4))
        for s in [2,3,4]:
            masks=[]
            max_n=(30+1)//(s+1)
            for n in range(3,max_n+1):
                D=n*(s+1)-1
                if D<=30:
                    m=torch.zeros(1,1,30,30)
                    m[:,:,:D,:D]=1.0
                    masks.append(m)
            self.register_buffer(f'masks{s}', torch.cat(masks,dim=1))
    def shiftN(self,x,dy:int,dx:int,N:int):
        zrow=getattr(self,f'z1_row_{N}')
        zcol=getattr(self,f'z1_col_{N}')
        if dy==0:
            y=x
        elif dy==1:
            y=torch.cat([zrow, x[:,:,:N-1,:]], dim=2)
        elif dy==-1:
            y=torch.cat([x[:,:,1:,:], zrow], dim=2)
        else:
            y=x*0.0
        if dx==0:
            return y
        elif dx==1:
            return torch.cat([zcol, y[:,:,:,:N-1]], dim=3)
        elif dx==-1:
            return torch.cat([y[:,:,:,1:], zcol], dim=3)
        else:
            return y*0.0
    def cell_copyN(self,C,N:int):
        occ=torch.clamp(C.sum(dim=1,keepdim=True),0,1)
        near=torch.zeros_like(occ)
        for dy in [-1,0,1]:
            for dx in [-1,0,1]:
                if dy or dx:
                    near=torch.clamp(near+self.shiftN(occ,dy,dx,N),0,1)
        active_cells=[]
        for k in range(9):
            mk=C[:,k:k+1]
            ctx=mk*near
            iso=mk*(1-near)
            active=((ctx.sum(dim=(2,3),keepdim=True)>0).float()*((iso.sum(dim=(2,3),keepdim=True)>0).float()))
            active_cells.append(mk*active)
        marker=torch.clamp(torch.cat(active_cells,dim=1).sum(dim=1,keepdim=True),0,1)
        ctx=marker*near
        outs=[]
        for j in range(9):
            cj=C[:,j:j+1]
            add=torch.zeros_like(cj)
            for dy in [-1,0,1]:
                for dx in [-1,0,1]:
                    if dy==0 and dx==0:
                        continue
                    pat=((ctx*self.shiftN(cj,-dy,-dx,N)).sum(dim=(2,3),keepdim=True)>0).float()
                    add=torch.clamp(add+self.shiftN(marker,dy,dx,N)*pat,0,1)
            outs.append(torch.clamp(cj+add,0,1))
        return torch.cat(outs,dim=1)
    def valid_mask(self,x,s:int):
        xc=x[:,1:]
        gates=[]
        max_n=(30+1)//(s+1)
        for n in range(3,max_n+1):
            D=n*(s+1)-1
            if D<=30:
                row_counts=xc[:,:,s:s+1,:D].sum(dim=(2,3),keepdim=True)
                col_counts=xc[:,:,:D,s:s+1].sum(dim=(2,3),keepdim=True)
                g=((row_counts>float(D)-0.5).float()*(col_counts>float(D)-0.5).float()).sum(dim=1,keepdim=True)
                gates.append((g>0.5).float())
        gate=torch.cat(gates,dim=1)
        return torch.clamp((gate*getattr(self,f'masks{s}')).sum(dim=1,keepdim=True),0,1)
    def candidate2(self,xp,x):
        C=F.max_pool2d(xp[:,1:],kernel_size=2,stride=3)
        Co=self.cell_copyN(C,10)
        pix=F.conv_transpose2d(Co,self.w2,stride=3,groups=9)
        pix=torch.cat([pix,self.z9_29_1],dim=3)
        pix=torch.cat([pix,self.z9_1_30],dim=2)
        return pix*self.valid_mask(x,2)
    def candidate3(self,xp,x):
        C=F.max_pool2d(xp[:,1:],kernel_size=3,stride=4)
        Co=self.cell_copyN(C,8)
        pix=F.conv_transpose2d(Co,self.w3,stride=4,groups=9)
        pix=pix[:,:,:30,:30]
        return pix*self.valid_mask(x,3)
    def candidate4(self,xp,x):
        C=F.max_pool2d(xp[:,1:],kernel_size=4,stride=5)
        Co=self.cell_copyN(C,6)
        pix=F.conv_transpose2d(Co,self.w4,stride=5,groups=9)
        pix=torch.cat([pix,self.z9_29_1],dim=3)
        pix=torch.cat([pix,self.z9_1_30],dim=2)
        return pix*self.valid_mask(x,4)
    def forward(self,x):
        xp=torch.cat([x,self.z10_col],dim=3)
        xp=torch.cat([xp,self.z10_row31],dim=2)
        out=x[:,1:]
        out=torch.clamp(out+self.candidate2(xp,x)+self.candidate3(xp,x)+self.candidate4(xp,x),0,1)
        active=(x.sum(dim=1,keepdim=True)>0.5).float()
        out=out*active
        mx=torch.clamp(out.sum(dim=1,keepdim=True),0,1)
        bg=active*(1.0-mx)
        return torch.cat([bg,out],dim=1)


In [7]:
with open(TASK_PATH) as f:
    task=json.load(f)
print({k:len(task.get(k,[])) for k in ['train','test','arc-gen']})

{'train': 3, 'test': 1, 'arc-gen': 262}


In [8]:

model=Task080StaticSafe().eval()
dummy=torch.zeros(1,CH,H,W,dtype=torch.float32)

torch.onnx.export(model,dummy,str(ONNX_PATH),input_names=['input'],
                  output_names=['output'],opset_version=13,
                  dynamic_axes=None,do_constant_folding=True,dynamo=False)

m=onnx.load(str(ONNX_PATH))
for vi in [m.graph.input[0], m.graph.output[0]]:
    dims=vi.type.tensor_type.shape.dim
    for d,v in zip(dims,[1,CH,H,W]):
        d.dim_param=''
        d.dim_value=int(v)
onnx.save(m,str(ONNX_PATH))
onnx.checker.check_model(m)

ops=op_counts(m)
forbidden={'Loop','Scan','NonZero','Unique','Script','Function'}
risk={'ScatterND','Shape','Range','Expand','Gather','ConstantOfShape'}
health={
    'task_id': TASK_ID,
    'input_shape': onnx_shape(m.graph.input[0]),
    'output_shape': onnx_shape(m.graph.output[0]),
    'size_bytes': ONNX_PATH.stat().st_size,
    'op_counts': ops,
    'forbidden_ops_present': sorted(forbidden.intersection(ops)),
    'risk_ops_present': sorted(risk.intersection(ops)),
    'accuracy': {}
}
health


/tmp/ipykernel_16/3330301352.py:4: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(model,dummy,str(ONNX_PATH),input_names=['input'],


{'task_id': 'task080',
 'input_shape': [1, 10, 30, 30],
 'output_shape': [1, 10, 30, 30],
 'size_bytes': 519620,
 'op_counts': {'Concat': 280,
  'Constant': 2290,
  'Slice': 344,
  'MaxPool': 3,
  'ReduceSum': 332,
  'Clip': 278,
  'Add': 270,
  'Mul': 568,
  'Sub': 4,
  'Greater': 322,
  'Cast': 322,
  'ConvTranspose': 3},
 'forbidden_ops_present': [],
 'risk_ops_present': [],
 'accuracy': {}}

In [9]:

sess=ort.InferenceSession(str(ONNX_PATH),providers=['CPUExecutionProvider'])
for split in ['train','test','arc-gen']:
    res=exact_eval(sess, task.get(split,[]))
    health['accuracy'][split]=res
    if res['first_wrong'] is not None:
        raise AssertionError((split,res))

valid_ag=valid_30_examples(task.get('arc-gen',[]))
cut=int(len(valid_ag)*0.4)
fit=exact_eval(sess, valid_ag[:cut])
hold=exact_eval(sess, valid_ag[cut:])
health['accuracy']['arc_gen_40_fit_60_holdout']={'fit':fit,'holdout':hold}


# Kaggle-safe raw one-hot audit: real grid is one-hot; padding outside real grid is all-zero.
def target_onehot(grid):
    y=np.zeros((1,CH,H,W),dtype=np.float32)
    a=np.array(grid,dtype=np.int64)
    for r in range(min(H,a.shape[0])):
        for cc in range(min(W,a.shape[1])):
            v=int(a[r,cc])
            if 0<=v<CH:
                y[0,v,r,cc]=1.0
    # padded area outside the real grid stays all-zero, matching Kaggle/reference contract
    return y

def raw_onehot_eval(sess, examples):
    inp=sess.get_inputs()[0].name
    exact=0; used=0; first=None
    for i,ex in enumerate(examples):
        if len(ex['input'])>H or len(ex['input'][0])>W or len(ex['output'])>H or len(ex['output'][0])>W:
            continue
        y=sess.run(None,{inp:grid_to_tensor(ex['input'])})[0]
        tgt=target_onehot(ex['output'])
        ok=np.array_equal(y,tgt)
        exact += int(ok); used += 1
        if not ok and first is None:
            first={'idx':i,'max_abs_diff':float(np.max(np.abs(y-tgt))),
                   'unique_output_values':[float(v) for v in np.unique(y)[:20]],
                   'channel_sum_values':[float(v) for v in np.unique(y.sum(axis=1))[:20]]}
    return {'exact':exact,'total':used,'first_wrong':first}

health['raw_onehot_accuracy']={}
for split in ['train','test']:
    rawres=raw_onehot_eval(sess, task.get(split,[]))
    health['raw_onehot_accuracy'][split]=rawres
    if rawres['first_wrong'] is not None:
        raise AssertionError(('raw_onehot', split, rawres))
raw_hold=raw_onehot_eval(sess, valid_ag[cut:])
health['raw_onehot_accuracy']['arc_gen_60_holdout']=raw_hold
assert raw_hold['exact']==raw_hold['total'] and raw_hold['total']>0

assert health['input_shape']==[1,10,30,30]
assert health['output_shape']==[1,10,30,30]
assert health['size_bytes']<1_400_000
assert not health['forbidden_ops_present']
assert not health['risk_ops_present']
assert hold['exact']==hold['total'] and hold['total']>0

with open(AUDIT_JSON,'w') as f:
    json.dump(health,f,indent=2)
with open(AUDIT_CSV,'w',newline='') as f:
    w=csv.writer(f)
    w.writerow(['task','onnx_size','input_shape','output_shape','forbidden_ops','risk_ops','train','test','arc_gen_valid','arc_gen_skipped_over_30','holdout60'])
    w.writerow([TASK_ID,health['size_bytes'],health['input_shape'],health['output_shape'],health['forbidden_ops_present'],health['risk_ops_present'],
                f"{health['accuracy']['train']['exact']}/{health['accuracy']['train']['total']}",
                f"{health['accuracy']['test']['exact']}/{health['accuracy']['test']['total']}",
                f"{health['accuracy']['arc-gen']['exact']}/{health['accuracy']['arc-gen']['total']}",
                health['accuracy']['arc-gen']['skipped_over_30'],
                f"{hold['exact']}/{hold['total']}"])
health


{'task_id': 'task080',
 'input_shape': [1, 10, 30, 30],
 'output_shape': [1, 10, 30, 30],
 'size_bytes': 519620,
 'op_counts': {'Concat': 280,
  'Constant': 2290,
  'Slice': 344,
  'MaxPool': 3,
  'ReduceSum': 332,
  'Clip': 278,
  'Add': 270,
  'Mul': 568,
  'Sub': 4,
  'Greater': 322,
  'Cast': 322,
  'ConvTranspose': 3},
 'forbidden_ops_present': [],
 'risk_ops_present': [],
 'accuracy': {'train': {'exact': 3,
   'total': 3,
   'skipped_over_30': 0,
   'first_wrong': None},
  'test': {'exact': 1, 'total': 1, 'skipped_over_30': 0, 'first_wrong': None},
  'arc-gen': {'exact': 227,
   'total': 227,
   'skipped_over_30': 35,
   'first_wrong': None},
  'arc_gen_40_fit_60_holdout': {'fit': {'exact': 90,
    'total': 90,
    'skipped_over_30': 0,
    'first_wrong': None},
   'holdout': {'exact': 137,
    'total': 137,
    'skipped_over_30': 0,
    'first_wrong': None}}},
 'raw_onehot_accuracy': {'train': {'exact': 3,
   'total': 3,
   'first_wrong': None},
  'test': {'exact': 1, 'total': 1

In [10]:
for zp in [ZIP_PATH, GENERIC_ZIP]:
    if zp.exists():
        zp.unlink()
    with zipfile.ZipFile(zp,'w',compression=zipfile.ZIP_DEFLATED) as z:
        z.write(ONNX_PATH,arcname=f'{TASK_ID}.onnx')
print('wrote', ZIP_PATH, 'and', GENERIC_ZIP)
print('zip contents:', zipfile.ZipFile(GENERIC_ZIP).namelist())

wrote /kaggle/working/task080_zero_pad_static_graph_submission.zip and /kaggle/working/submission.zip
zip contents: ['task080.onnx']
